# Question 1

In [1]:
from datasets import load_dataset

# Load the train.csv dataset
dataset = load_dataset('csv', data_files={'train': '/content/train.csv'})

# Access the training split
train_dataset = dataset['train']

# Define a function to concatenate prompt and column A
def concatenate_text(example):
    # Ensure both 'prompt' and 'A' columns exist and are not None
    prompt_text = str(example.get('prompt', ''))
    a_text = str(example.get('A', ''))
    example['combined_text'] = f"{prompt_text} {a_text}"
    return example

# Apply the function to create the new 'combined_text' column
train_dataset = train_dataset.map(concatenate_text)

# Get the combined_text string for the row at index 51
combined_text_51 = train_dataset[51]['combined_text']

# Calculate the character length
character_length = len(combined_text_51)

print(f"The character length for combined_text at index 51 is: {character_length}")

The character length for combined_text at index 51 is: 614


# Question 2

In [2]:
from transformers import AutoTokenizer

# Initialize the bert-base-uncased tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Get the total vocabulary size
vocabulary_size = tokenizer.vocab_size

print(f"The total vocabulary size is: {vocabulary_size}")

The total vocabulary size is: 30522


# Question 3

In [3]:
# Get the integer ID for the [SEP] token
sep_token_id = tokenizer.sep_token_id

print(f"The integer ID assigned to the [SEP] token: {sep_token_id}")

The integer ID assigned to the [SEP] token: 102


# Question 4

In [4]:
# Extract the prompt column from the train_dataset and convert to a Python list
prompts = list(train_dataset['prompt'])

# Tokenize the entire prompt column simultaneously
tokenized_prompts = tokenizer(
    prompts,
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

# Get the shape of the input_ids tensor
input_ids_shape = tokenized_prompts['input_ids'].shape

print(f"The shape of the resulting tensor is: {input_ids_shape}")

The shape of the resulting tensor is: torch.Size([2000, 128])


# Question 5

In [5]:
# Given parameters
hidden_embedding_size = 768
num_attention_heads = 12

# Calculate the dimensionality of each individual attention head
attention_head_dimensionality = hidden_embedding_size // num_attention_heads

print(f"The dimensionality of each attention head is: {attention_head_dimensionality}")

The dimensionality of each attention head is: 64


# Question 6

In [6]:
from transformers import AutoModel
import torch

# Load the bert-base-uncased model
model = AutoModel.from_pretrained('bert-base-uncased')

# Get the prompt from row ID 0
prompt_row_0 = train_dataset[0]['prompt']

# Tokenize the prompt using default settings
tokenized_input = tokenizer(prompt_row_0, return_tensors='pt')

# Pass the tokenized input through the model
with torch.no_grad(): # Disable gradient calculations for inference
    outputs = model(**tokenized_input)

# Get the last_hidden_state tensor
last_hidden_state = outputs.last_hidden_state

# Get the shape of the last_hidden_state tensor
last_hidden_state_shape = last_hidden_state.shape

print(f"The shape of the last_hidden_state tensor is: {last_hidden_state_shape}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


The shape of the last_hidden_state tensor is: torch.Size([1, 31, 768])


# Question 7

In [7]:
# Extract the [CLS] token embedding (always at index 0)
cls_embedding = last_hidden_state[0, 0, :]

# Get the first 5 float values
first_5_values = cls_embedding[:5]

# Calculate the sum of the first 5 values
sum_first_5 = torch.sum(first_5_values).item()

# Round the sum to 4 decimal places
rounded_sum = round(sum_first_5, 4)

print(f"The sum of the first 5 float values in the [CLS] vector: {rounded_sum}")

The sum of the first 5 float values in the [CLS] vector: -1.2001


# Question 8

In [8]:
# Load the bert-base-uncased model with output_attentions=True
model_with_attentions = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True)

# Define the exact string to tokenize
sentence = "Light-ion fusion is a technique."

# Tokenize the sentence
tokenized_sentence = tokenizer(sentence, return_tensors='pt')

# Pass the tokenized input through the model
with torch.no_grad():
    outputs_with_attentions = model_with_attentions(**tokenized_sentence)

# Extract the attention matrix for the last layer (index -1)
last_layer_attentions = outputs_with_attentions.attentions[-1]

# Extract the attention matrix for the first attention head (head index 0)
first_head_attention = last_layer_attentions[0, 0, :, :]

# Find the token index for 'fusion'
input_ids = tokenized_sentence['input_ids'][0]
decoded_tokens = tokenizer.convert_ids_to_tokens(input_ids)

fusion_token_index = -1
for i, token in enumerate(decoded_tokens):
    if 'fusion' in token: # Check for 'fusion' or '##fusion' if it's a subword
        fusion_token_index = i
        break

if fusion_token_index == -1:
    print(f"Warning: 'fusion' token not found or split. Decoded tokens: {decoded_tokens}")
    fusion_token_index = decoded_tokens.index('fusion')

# The [CLS] token is always at index 0
cls_token_index = 0

# Get the attention weight from [CLS] (query) to 'fusion' (key)
attention_weight = first_head_attention[cls_token_index, fusion_token_index].item()

# Round the answer to 4 decimal places
rounded_attention_weight = round(attention_weight, 4)

print(f"Decoded tokens: {decoded_tokens}")
print(f"Index of 'fusion' token: {fusion_token_index}")
print(f"The attention weight from [CLS] to 'fusion': {rounded_attention_weight}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Decoded tokens: ['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
Index of 'fusion' token: 4
The attention weight from [CLS] to 'fusion': 0.1025


# Question 9

In [9]:
from sentence_transformers import SentenceTransformer, util

# Initialize the sentence-transformers/all-MiniLM-L6-v2 model
sentence_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Get the prompt and Option B for row ID 0
prompt_0 = train_dataset[0]['prompt']
option_b_0 = train_dataset[0]['B']

# Generate embeddings for both
prompt_embedding = sentence_model.encode(prompt_0, convert_to_tensor=True)
option_b_embedding = sentence_model.encode(option_b_0, convert_to_tensor=True)

# Calculate the cosine similarity
cosine_similarity = util.cos_sim(prompt_embedding, option_b_embedding).item()

# Round the similarity score to 4 decimal places
rounded_similarity = round(cosine_similarity, 4)

print(f"The cosine similarity between the prompt and Option B for row 0: {rounded_similarity}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

The cosine similarity between the prompt and Option B for row 0: 0.7658


# Question 10

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, util
from tqdm.auto import tqdm
import numpy as np

# Pipeline 1
# Create a DataFrame for easier processing
train_df = train_dataset.to_pandas()

# Prepare options for TF-IDF
option_columns = ['A', 'B', 'C', 'D', 'E']

# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer()

# Fit on all prompts and options to build a comprehensive vocabulary
corpus = list(train_df['prompt']) + train_df[option_columns].values.flatten().tolist()
tfidf_vectorizer.fit(corpus)

def get_tfidf_rankings(row_id):
    prompt = train_df.loc[row_id, 'prompt']
    correct_answer_label = train_df.loc[row_id, 'answer']
    options = {col: train_df.loc[row_id, col] for col in option_columns}

    prompt_vec = tfidf_vectorizer.transform([prompt])

    similarities = []
    for label, option_text in options.items():
        option_vec = tfidf_vectorizer.transform([option_text])
        sim = cosine_similarity(prompt_vec, option_vec)[0][0]
        similarities.append({'label': label, 'similarity': sim})

    # Sort by similarity in descending order
    ranked_options = sorted(similarities, key=lambda x: x['similarity'], reverse=True)

    # Get Top-3 labels
    top_3_labels = [opt['label'] for opt in ranked_options[:3]]

    return top_3_labels, correct_answer_label

# Pipeline 2
# Re-initialize the MiniLM model
sentence_model_minilm = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

def get_minilm_rankings(row_id):
    prompt = train_df.loc[row_id, 'prompt']
    correct_answer_label = train_df.loc[row_id, 'answer']
    options = {col: train_df.loc[row_id, col] for col in option_columns}

    prompt_embedding = sentence_model_minilm.encode(prompt, convert_to_tensor=True)

    similarities = []
    for label, option_text in options.items():
        option_embedding = sentence_model_minilm.encode(option_text, convert_to_tensor=True)
        sim = util.cos_sim(prompt_embedding, option_embedding).item()
        similarities.append({'label': label, 'similarity': sim})

    # Sort by similarity in descending order
    ranked_options = sorted(similarities, key=lambda x: x['similarity'], reverse=True)

    # Get Top-3 labels
    top_3_labels = [opt['label'] for opt in ranked_options[:3]]

    return top_3_labels, correct_answer_label

def calculate_map3(predictions_list):
    ap_scores = []
    for top_3, correct_ans in predictions_list:
        precision_at_k = []
        for k in range(len(top_3)):
            if top_3[k] == correct_ans:
                # If correct answer is found at rank k+1, precision is (number of correct up to k+1) / (k+1)
                precision_at_k.append(1 / (k + 1))
            else:
                precision_at_k.append(0)

        # Average Precision for this query
        if correct_ans in top_3:
            correct_rank = top_3.index(correct_ans) + 1
            ap = sum([1 / i for i in range(1, correct_rank + 1) if top_3[i-1] == correct_ans]) / 1 # divided by 1 because there is only one relevant item (the correct answer)
            ap_scores.append(ap)
        else:
            ap_scores.append(0)

    return np.mean(ap_scores)


tfidf_predictions = []
minilm_predictions = []

for i in tqdm(range(len(train_df)), desc="Processing rows for ranking"): # Use tqdm for progress bar
    tfidf_top3, tfidf_correct_ans = get_tfidf_rankings(i)
    minilm_top3, minilm_correct_ans = get_minilm_rankings(i)

    tfidf_predictions.append((tfidf_top3, tfidf_correct_ans))
    minilm_predictions.append((minilm_top3, minilm_correct_ans))

# Calculate MAP@3 for MiniLM pipeline
map3_minilm = calculate_map3(minilm_predictions)
print(f"Final MAP@3 score of the all-MiniLM-L6-v2 pipeline: {round(map3_minilm, 4)}")

# Count questions where correct answer is NOT in TF-IDF Top-3 BUT IS in MiniLM Top-3
count_exclusive_minilm_wins = 0
for i in range(len(train_df)):
    tfidf_top3, correct_ans_tfidf = tfidf_predictions[i]
    minilm_top3, correct_ans_minilm = minilm_predictions[i]

    # Ensure correct answers are the same for comparison
    if correct_ans_tfidf != correct_ans_minilm:
        print(f"Warning: Mismatch in correct answer for row {i}")
        continue

    if correct_ans_tfidf not in tfidf_top3 and correct_ans_minilm in minilm_top3:
        count_exclusive_minilm_wins += 1

print(f"Required number of questions: {count_exclusive_minilm_wins}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Processing rows for ranking:   0%|          | 0/2000 [00:00<?, ?it/s]

Final MAP@3 score of the all-MiniLM-L6-v2 pipeline: 0.4231
Required number of questions: 531


# Question 11

In [12]:
from transformers import pipeline

# Initialize the zero-shot-classification pipeline
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Get the prompt for the 2nd row (index 1)
prompt_idx_1 = train_dataset[1]['prompt']

# Get Options A, B, and C for the 2nd row (index 1)
option_a_1 = train_dataset[1]['A']
option_b_1 = train_dataset[1]['B']
option_c_1 = train_dataset[1]['C']

candidate_labels = [option_a_1, option_b_1, option_c_1]

# Perform zero-shot classification
result = classifier(prompt_idx_1, candidate_labels)

# The top-ranked option is the first one in the 'labels' list
top_ranked_score = result['scores'][0]

# Round to 4 decimal places
rounded_top_score = round(top_ranked_score, 4)

print(f"The probability score for the top-ranked option is: {rounded_top_score}")

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

The probability score for the top-ranked option is: 0.4575


# Question 12

In [13]:
from transformers import pipeline
import numpy as np

sum_probs_softmax = sum(result['scores'])
# Perform zero-shot classification with multi_label=True
result_multilabel = classifier(prompt_idx_1, candidate_labels, multi_label=True)

# Get the probabilities for the multi_label run
sum_probs_sigmoid = sum(result_multilabel['scores'])

# Calculate the absolute difference
absolute_difference = abs(sum_probs_sigmoid - sum_probs_softmax)

# Round to 4 decimal places
rounded_absolute_difference = round(absolute_difference, 4)

print(f"Sum of probabilities (Softmax, multi_label=False): {sum_probs_softmax:.4f}")
print(f"Sum of probabilities (Sigmoid, multi_label=True): {sum_probs_sigmoid:.4f}")
print(f"The absolute difference between the sum of the 3 probabilities (Softmax vs. Sigmoid) is: {rounded_absolute_difference}")

Sum of probabilities (Softmax, multi_label=False): 1.0000
Sum of probabilities (Sigmoid, multi_label=True): 0.0005
The absolute difference between the sum of the 3 probabilities (Softmax vs. Sigmoid) is: 0.9995


# Question 13

In [15]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load the tokenizer and model directly for Flan-T5 (a sequence-to-sequence model)
tokenizer_flan = AutoTokenizer.from_pretrained("google/flan-t5-small")
model_flan = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

# Get the prompt, Option A, and Option B for row index 0
prompt_idx_0 = train_dataset[0]['prompt']
option_a_idx_0 = train_dataset[0]['A']
option_b_idx_0 = train_dataset[0]['B']

# Construct the exact input string
input_string = f"Question: {prompt_idx_0}. Is the correct answer A: {option_a_idx_0} or B: {option_b_idx_0}? Answer with just the letter A or B."

# Tokenize the input string
input_ids = tokenizer_flan(input_string, return_tensors="pt").input_ids

# Generate output using the model's generate method
generated_ids = model_flan.generate(input_ids, max_new_tokens=5)

# Decode the generated tokens back to a string
generated_output = tokenizer_flan.decode(generated_ids[0], skip_special_tokens=True)

print(f"Input string to the model: {input_string}")
print(f"Exact string output returned by the model: '{generated_output}'")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Input string to the model: Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.. Is the correct answer A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time. or B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.? Answer with just the letter A or B.
Exact string output returned by the model: 'B'
